# Machine Learning Project for California Housing 
Based on _Hands-on Machine Learning_ by Aurelien Geron


In [ ]:
from sklearn.model_selection import train_test_split
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


## Load Data

In [ ]:
housing_full = pd.read_csv(Path("data/housing.csv"))

## Visualise Data

In [ ]:
plt.rc("font", size=8)
housing_full.hist(bins=50, figsize=(12, 8))

Observations: 
- median age and median income are capped. This may be wrongly learnt by the model
- many features are right-skewed, which is not good for machine learning
- total_bedrooms have null values 
- different features have very different scales

## Train Test Split
- you can pass it multiple datasets with an identical number of rows, and it will split them on the same indices
- splitting randomly here is fine, but it may not be when population size is small
- here we are creating a new feature `income_cat` for use in stratified sampling

In [ ]:
housing_full["income_cat"] = pd.cut(housing_full["median_income"], 
                                    bins=[0., 1.5, 3.0, 4.5, 6.0, np.inf],
                                    labels=[1, 2, 3, 4, 5])


Visualisation of income_cat

In [ ]:
housing_full["income_cat"].value_counts().sort_index().plot(kind="bar")
plt.show()

Comparing stratified sampling and random sampling
- apparently the composition of test_1 is closer to housing_full

In [ ]:
# splitting randomly 
train, test = train_test_split(housing_full, test_size=0.2, random_state=42)

# splitting with stratified sampling 
train_1, test_1 = train_test_split(housing_full, test_size=0.2, random_state=42, stratify=housing_full["income_cat"])

# print(test["income_cat"].value_counts().sort_index() / len(test))
# print(test_1["income_cat"].value_counts().sort_index() / len(test))
# print(housing_full["income_cat"].value_counts().sort_index() / len(housing_full))

# now income_cat is useless, we drop it 
for set in (train_1, test_1): 
    set.drop("income_cat", axis=1, inplace=True)

# make a copy of train_1 for future use 
housing: pd.DataFrame = train_1.copy()

## Data Visualisation
Visualise geographical data

In [ ]:
housing.plot(kind="scatter", x="longitude", y="latitude", grid=True, alpha=0.2)
plt.show()


Check correlations
- median house value is not correlated to population. Why? 

In [ ]:
corr_matrix = housing.corr(numeric_only=True)
corr_matrix["median_house_value"].sort_values(ascending=False)


Create scatter matrix

In [ ]:
from pandas.plotting import scatter_matrix
useful_attributes = ["median_house_value", "median_income", "total_rooms", "housing_median_age"]
scatter_matrix(housing[useful_attributes])
plt.show()

Plot a scatter of median house value and median income

In [ ]:
housing.plot(kind="scatter", x="median_income", y="median_house_value", alpha=0.2)

## Combine Attributes
- more rooms per house >> higher value
- a larger percentage of rooms are bedrooms >> lower value

In [ ]:
housing["rooms_per_house"] = housing["total_rooms"] / housing["households"]
housing["bedrooms_ratio"] = housing["total_bedrooms"] / housing["total_rooms"]
housing["people_per_house"] = housing["population"] / housing["households"]

useful_attributes_1 = ["rooms_per_house", "bedrooms_ratio", "people_per_house", "median_house_value"]
new_corr_matrix = housing[useful_attributes_1].corr()
print(new_corr_matrix["median_house_value"].sort_values(ascending=False))

## Prepare the data
- write functions instead of doing it manually
- so that the transformation is easily reproducible

In [ ]:
# start with a clean set of training data
housing = train_1.drop("median_house_value", axis=1)
housing_y = train_1["median_house_value"].copy()

Fill in missing total_bedrooms data
- Can use .fillna(), but better to use SimpleImputer
- It stores median value, which can be used on not only the training set

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

# median can only be computed for numerical values
# so we drop ocean_proximity
housing_num = housing.select_dtypes(include=[np.number])
imputer.fit(housing_num)
X = imputer.transform(housing_num)

# X is a numpy array, not a pandas dataframe 
housing_tr = pd.DataFrame(X, columns=housing_num.columns, 
                          index=housing_num.index)
housing_tr.info()


Convert categorical attributes into numbers

In [ ]:
ocean_prox = housing[["ocean_proximity"]].copy()

from sklearn.preprocessing import OrdinalEncoder
ordinal_encoder = OrdinalEncoder()
ocean_prox_ordinal = ordinal_encoder.fit_transform(ocean_prox)

# the problem with ordinal encoder is, the model will assume that 
# closer values are more similar, but that is not the case here
# so we use one hot encoder
from sklearn.preprocessing import OneHotEncoder
onehot_encoder = OneHotEncoder()
ocean_prox_onehot = onehot_encoder.fit_transform(ocean_prox)
ocean_prox_df = pd.DataFrame(ocean_prox_onehot.toarray(), columns=onehot_encoder.get_feature_names_out(), index=housing.index)


In [ ]:
housing.sort_values("median_income", ascending=False).head()

Feature Scaling
- Total number of rooms ranges from 2 to 39320
- Median income ranges from 0.49 to 15.1
- Without feature scaling, the model will prioritise total number of rooms 

**WARNING:**
- ONLY fit on training data. DO NOT fit on validation or test data. 

In [ ]:
from sklearn.preprocessing import StandardScaler

std_scaler = StandardScaler().set_output(transform="pandas")
housing_tr_scaled = std_scaler.fit_transform(housing_tr)

Notice that population is skewed to the right (i.e. have a long right tail)  
We want to make it symmetrical
- take square root
- or take logarithm

In [ ]:
fig, axes = plt.subplots(1, 2)
axes[0].hist(housing["population"], bins=50)
axes[1].hist(np.log(housing["population"]), bins=50)
plt.show()

If you have a multi-modal distribution (multiple peaks)
- housing median age has two peaks, one at 16 and another at 35
- Can split the data into bins and treat the feature as a categorical data
    - e.g. split housing median age into new, middle, old
- Or use RBF to measure the similarity between all values and a fixed point (the peak)


In [ ]:
from sklearn.metrics.pairwise import rbf_kernel

housing["housing_median_age"].hist(bins=50)
age_simil_35 = rbf_kernel(housing[["housing_median_age"]], [[35]], 0.1)

TransformedTargetRegressor  

Sometimes the labels (y) also need to be transformed. After you obtain predictions,  
you need to revert the transformed predictions back. 

I can also define my own transformer, such as a log transformer

In [ ]:
from sklearn.preprocessing import FunctionTransformer

log_transformer = FunctionTransformer(np.log, inverse_func=np.exp)
log_pop = log_transformer.transform(housing[["population"]])

## Transformation Pipeline

In [ ]:
from sklearn.pipeline import Pipeline 

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("standardise", StandardScaler()), 
])
num_pipeline.set_output(transform="pandas")
housing_num_prepared = num_pipeline.fit_transform(housing_num)


Make one pipeline that handles both categorical and numerical attributes

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
num_attribs = ["longitude", "latitude", "housing_median_age", "total_rooms",
"total_bedrooms", "population", "households", "median_income"]

cat_attribs = ["ocean_proximity"]

cat_pipeline = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore"))

preprocessing = ColumnTransformer([
    ("num", num_pipeline, num_attribs),
    ("cat", cat_pipeline, cat_attribs),
])

### Put everything in one pipeline

Create a FunctionTransformer that computes new features like rooms_per_house and bedroom_ratio

In [ ]:
def column_ratio(X):
    return X[:, [0]] / X[:, [1]]
def ratio_name(function_transformer, feature_names_in):
    return ["ratio"] # feature names out
def ratio_pipeline():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        FunctionTransformer(column_ratio, feature_names_out=ratio_name),
        StandardScaler())

Log transformer

In [ ]:
log_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    FunctionTransformer(np.log, feature_names_out="one-to-one"), 
    StandardScaler())

Cluster Similarity
- Measure how close one sample (row) is to the centre of a cluster

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import rbf_kernel

class ClusterSimilarity(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=10, gamma=1.0, random_state=None):
        self.n_clusters = n_clusters
        self.gamma = gamma
        self.random_state = random_state

    def fit(self, X, y=None, sample_weight=None):
        self.kmeans_ = KMeans(self.n_clusters, random_state=self.random_state)
        self.kmeans_.fit(X, sample_weight=sample_weight)
        return self

    def transform(self, X):
        return rbf_kernel(X, self.kmeans_.cluster_centers_, gamma=self.gamma)

    def get_feature_names_out(self, names=None):
        return [f"Cluster {i} similarity" for i in range(self.n_clusters)]

Final pipeline

In [ ]:
from sklearn.compose import make_column_selector

cluster_simil = ClusterSimilarity(n_clusters=10, gamma=1., random_state=42)
default_num_pipeline = make_pipeline(SimpleImputer(strategy="median"), 
                                     StandardScaler())   

preprocessing = ColumnTransformer([
    ("bedrooms", ratio_pipeline(), ["total_bedrooms", "total_rooms"]),
    ("rooms_per_house", ratio_pipeline(), ["total_rooms", "households"]),
    ("people_per_house", ratio_pipeline(), ["population", "households"]),
    ("log", log_pipeline, ["total_bedrooms", "total_rooms", "population",
    "households", "median_income"]),
    ("geo", cluster_simil, ["latitude", "longitude"]),
    ("cat", cat_pipeline, make_column_selector(dtype_include=object)),
],
remainder=default_num_pipeline) # one column remaining: housing_median_age

In [ ]:
housing_prepared = preprocessing.fit_transform(housing)
preprocessing.get_feature_names_out()

## Select and train the model

In [ ]:
from sklearn.linear_model import LinearRegression

lin_pipeline = make_pipeline(preprocessing, LinearRegression())
lin_pipeline.fit(housing, housing_y)
housing_predictions = lin_pipeline.predict(housing)
print(housing_predictions[:5].round(-2))
housing_y.iloc[:5]

Measure performance with using test labels 

In [ ]:
from sklearn.metrics import root_mean_squared_error

lin_error = root_mean_squared_error(housing_y, housing_predictions)
print(lin_error)


Decision tree regressor

In [ ]:
from sklearn.tree import DecisionTreeRegressor

tree_reg = make_pipeline(preprocessing, DecisionTreeRegressor(random_state=42))
tree_reg.fit(housing, housing_y)
tree_predictions = tree_reg.predict(housing)

tree_error = root_mean_squared_error(housing_y, tree_predictions)
tree_error


### Cross Validation

In [ ]:
from sklearn.model_selection import cross_val_score

tree_rmses = -cross_val_score(tree_reg, housing, housing_y, scoring="neg_root_mean_squared_error", cv=10)

Random forest regressor

In [ ]:
from sklearn.ensemble import RandomForestRegressor

forest_reg = make_pipeline(preprocessing, RandomForestRegressor(random_state=42))
# forest_rmses = -cross_val_score(forest_reg, housing, housing_y, scoring="neg_root_mean_squared_error", cv=10)

forest_reg.fit(housing, housing_y)

forest_predictions = forest_reg.predict(housing)

forest_rmse = root_mean_squared_error(housing_y, forest_predictions)
print(forest_rmse) 

# forest_rmse = 17519.685029292894
# this is the rmse on training set, which is lower than rmse on validation set
# meaning there is overfitting to the training set


17519.685029292894


## Hyperparameters tuning

Grid search

In [ ]:
from sklearn.model_selection import GridSearchCV

full_pipeline = Pipeline([("preprocessing", preprocessing), 
                          ("random_forest", RandomForestRegressor(random_state=42))])

param_grid = [
    {'preprocessing__geo__n_clusters': [5, 8, 10],
    'random_forest__max_features': [4, 6, 8]},
    {'preprocessing__geo__n_clusters': [10, 15],
    'random_forest__max_features': [6, 8, 10]},
]
grid_search = GridSearchCV(full_pipeline, param_grid, cv=3, scoring='neg_root_mean_squared_error')
grid_search.fit(housing, housing_y)